# Data Processing

## Imports

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
import re

from pathlib import Path

spark = SparkSession.builder.getOrCreate()

## Load

In [0]:
data_path = Path.cwd().parent / 'data' / 'raw'

profile_schema = StructType([
    StructField("age", LongType(), True),
    StructField("credit_card_limit", DoubleType(), True),
    StructField("gender", StringType(), True),
    StructField("id", StringType(), True),
    StructField("registered_on", DateType(), True)
])

offers = (
    spark.read.json(str(data_path / 'offers.json'))
    .withColumnRenamed("id", "offer_id")
)
profile = (
    spark.read.json(str(data_path / 'profile.json'), schema=profile_schema, dateFormat="yyyyMMdd")
    .withColumn("age", F.when(F.col("age")!=118, F.col("age")))
    .withColumnRenamed("id", "account_id")
)
transactions = (
    spark.read.json(str(data_path / 'transactions.json'))
    .withColumn("offer_id", F.coalesce(F.col("value.offer id"), F.col("value.offer_id")))
    .withColumn("transaction_amount", F.col("value.amount"))
    .withColumn("offer_reward", F.col("value.reward"))
    .drop("value")
)


# TODO
# Remover accounts que tem offer, porem não tem o ID

## Validations

### Offers

In [0]:
offers.count()

In [0]:
offers.printSchema()

In [0]:
offers.orderBy("offer_type", "channels").show(10, truncate=False)

In [0]:
(
    offers
    .groupBy("channels")
    .count()
).show(truncate=False)

### Profile

In [0]:
profile.count()

In [0]:
profile.printSchema()

In [0]:
profile.show()

In [0]:
profile.groupBy("age").count().orderBy("age").display()

Databricks visualization. Run in Databricks to view.

In [0]:
profile.agg(F.mean(F.col("age").isNull().cast("int"))).show()

In [0]:
(
    profile
    .groupBy(
        F.col("age") == 118,
        F.col("credit_card_limit").isNull(),
        F.col("gender").isNull()
    )
    .count()
).show()

In [0]:
profile.groupby("gender").count().show()

In [0]:
profile.select("credit_card_limit").display()

Databricks visualization. Run in Databricks to view.

### Transactions

In [0]:
transactions.count()

In [0]:
transactions.printSchema()

In [0]:
transactions.orderBy(F.rand()).display()

In [0]:
display(
    transactions
    .where(F.col("account_id") == "78afa995795e4d85b5d9ceeca43f5fef")
    .orderBy("time_since_test_start")
    .join(offers, "offer_id", "left")
)

In [0]:
transactions.groupBy("event").count().show()

In [0]:
display(
    transactions
    .where(F.col("event") != "transaction")
    .groupBy("account_id", "offer_id")
    .agg(
        F.collect_list("event").alias("list_events")
    )
    .groupBy("list_events")
    .count()
    .orderBy(F.col("count").desc())
)

In [0]:
validacao_df = (transactions
    .filter(F.col("event").isin("offer received", "offer viewed"))
    .groupBy("account_id", "offer_id")
    .pivot("event")
    .count()
    .na.fill(0)
)

casos_multiplos = validacao_df.filter(F.col("offer viewed") > F.col("offer received"))

casos_multiplos.show(10, truncate=False)

In [0]:
# Ninguem recebeu mais de uma offer no mesmo dia
# transactions
#     .where(F.col("event") == "offer received")
#     .withColumn("rank", F.rank().over(Window.partitionBy("account_id").orderBy("time_since_test_start")))
#     .groupBy("account_id", "rank")
#     .count()
#     .where(F.col("count")>1)

# Considerando todas as variantes do desconto temos cerca de 1 account por desconto
# df = (
#     transactions
#     .where(F.col("event") == "offer received")
#     .withColumn("rank", F.rank().over(Window.partitionBy("account_id").orderBy("time_since_test_start")))
#     .join(offers, "offer_id", "left")
#     .groupBy("account_id")
#     .pivot("rank")
#     .agg(
#         F.min("time_since_test_start").alias("time"),
#         F.min("discount_value").alias("discount"),
#         F.min("duration").alias("duration"),
#         F.min("min_value").alias("min"),
#         F.first("channels").alias("channel")
#     )
# )

# cols = df.columns
# cols = {i : i.replace("\d_", "") for i in cols}


# cols_mapping = {}

# for col in df.columns:
#     match = re.match(r'^(\d+)_(.+)$', col)
#     if match:
#         number = match.group(1)
#         text = match.group(2)
    
#         new_name = f"{text}_{number}"
#         cols_mapping[col] = new_name
#     else:
#         cols_mapping[col] = col

# df = df.withColumnsRenamed(cols_mapping)

# display(
#     df
#     .groupBy(
#         *[i for i in cols_mapping.values() if i != "account_id"]
#     )
#     .count()
#     .orderBy(F.col("count").desc())
#

# Offers são enviadas no dia 0, 7, 14, 17 e 21
# display(
#     transactions
#     .where(F.col("event") == "offer received")
#     .withColumn("rank", F.rank().over(Window.partitionBy("account_id").orderBy("time_since_test_start")))
#     .join(offers, "offer_id", "left")
#     .groupBy("account_id")
#     .pivot("rank")
#     .agg(
#         F.min("time_since_test_start").alias("time"),
#         F.min("discount_value").alias("discount"),
#         F.min("duration").alias("duration"),
#         F.min("min_value").alias("min"),
#         F.first("channels").alias("channel")
#     )
# )

(
    transactions
    .where(F.col("event") == "offer received")
    .withColumnRenamed("time_since_test_start", "received_at")
    # .join(
    #     transactions
    #     .where(F.col("event") == "offer viewed")
    #     .select(
    #         F.col("account_id"),
    #         F.col("offer_id"),
    #         F.col("time_since_test_start").alias("viewed_at")
    #     ),
    #     ['account_id', "offer_id"],
    #     "left"
    # )
    .join(
        transactions
        .where(F.col("event") == "offer completed")
        .select(
            F.col("account_id"),
            F.col("offer_id"),
            F.col("time_since_test_start").alias("completed_at")
        ),
        ['account_id', "offer_id"],
        "left"
    )
    .count()
    # .withColumn("rank", F.rank().over(Window.partitionBy("account_id").orderBy("time_since_test_start")))
    # .join(offers, "offer_id", "left")
    # .groupBy("account_id")
    # .pivot("rank")
    # .agg(
    #     F.min("time_since_test_start").alias("time"),
    #     F.min("discount_value").alias("discount"),
    #     F.min("duration").alias("duration"),
    #     F.min("min_value").alias("min"),
    #     F.first("channels").alias("channel")
    # )
)

In [0]:
transactions.orderBy(F.rand()).select("time_since_test_start").display()

Databricks visualization. Run in Databricks to view.

In [0]:
transactions.select("time_since_test_start").describe().show()

In [0]:
display(
    transactions
    .groupBy("account_id")
    .agg(
        F.count("*"),
        F.min("time_since_test_start"),
        F.max("time_since_test_start")
    )
)

Databricks visualization. Run in Databricks to view.

In [0]:
# Todas as offers estão da transactions
display(
    transactions
    .where(F.col("offer_id").isNotNull())
    .select(F.col("offer_id"), F.lit(1).alias("in_transactions"))
    .join(
        offers
        .select(
            F.col("offer_id"), F.lit(1).alias("in_offers")
        ), 
    ['offer_id'], "full")
    .distinct()
)

In [0]:
# Todas as accounts estão na transactions
display(
    transactions
    .where(F.col("account_id").isNotNull())
    .select(F.col("account_id"), F.lit(1).alias("in_transactions"))
    .join(
        profile
        .select(
            F.col("account_id"), F.lit(1).alias("in_profile")
        ), 
    ['account_id'], "full")
    .distinct()
    .agg(
        F.count("*"),
        F.sum("in_transactions"),
        F.sum("in_profile")
    )
)

In [0]:
# 6 accounts não receberam offers

display(
    transactions
    .groupBy("account_id")
    .agg(
        F.sum(F.when(F.col("event")=="offer received", 1)).alias("offer_received"),
        F.sum(F.when(F.col("event")=="offer viewed", 1)).alias("offer_viewed"),
        F.sum(F.when(F.col("event")=="offer completed", 1)).alias("offer_completed")
    )
    .groupBy("offer_received")
    .count()
)

In [0]:
# Temos um grupo controle?

display(
    transactions
    .where(F.col("event")=="offer received")
    .join(
        transactions
        .where(F.col("event")=="offer received")
        .where(F.col("offer_id").isin("3f207df678b143eea3cee63160fa8bed", "5a8bc65990b245e5a138643cd4eb9837"))
        .select("account_id"),
        "account_id", "inner"
    )
    .groupBy("account_id")
    .agg(
        F.collect_set("offer_id").alias("offer_ids"),
    )
    .withColumn("size", F.size(F.col("offer_ids")))
)

Databricks visualization. Run in Databricks to view.